# H0 · Reference Ship Database
**Ship and Ocean Systems Design 2026 · Homework H0 · optional, not graded · individual**

Automates **Exercise 1.4** (Lecture 1.4, slide 11, solved in class) on **the reference ship**, the 37,600 t oil/chemical tanker (owner's brief: 0.6 The Reference Ship, slide 2). Practice for D2. Papanikolaou (2026), §1.3.6–1.3.7, §2.1–2.2, Table 2.15 and Appendix A.4.

What this notebook does:
1. Estimates Δ from the deadweight: the DWT/Δ range of Table 2.1 and the two tanker fits, Figs. A.51 (ratio) and A.52 (linear).
2. Reads the **tanker** regressions of Appendix A.4 (Figs. A.53–A.58) at the reference ship's DWT, each with an approximate **95 % prediction band** read off the figures.
3. Takes the block coefficient from Horn's formula (Table 2.15), since the tanker fits give none.
4. Runs the checks of Lecture 1.4 (closure, lightship, ratios, data range) and the owner's draught limit.
5. Plots length against deadweight with its band.

Compare your numbers with **Exercise Answers – Week 1**. Run it in Google Colab or VS Code. Change only the cells marked **INPUT**.

In [ ]:
# INPUT: the reference ship (owner's brief, 0.6 slide 2)
SHIP      = "Reference ship"
DWT       = 37_600.0           # deadweight [t]
V_SERVICE = 14.5               # service speed [kn]
T_MAX     = 11.5               # draught limit at Sines and Rotterdam [m] (course assumption in the brief)

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RHO, G, KN = 1.025, 9.81, 0.514444
print(f"{SHIP}: DWT = {DWT:,.0f} t, V = {V_SERVICE} kn, draught limit {T_MAX} m")

## 1 · Δ from the deadweight
Table 2.1 gives DWT/Δ = 0.78–0.88 for crude oil tankers (Lecture 1.4). The tanker fits of Appendix A.4 give Δ in two ways: through the ratio DWT/Δ (Fig. A.51) and directly, as a straight line (Fig. A.52). Two fits of the same data, two answers.

In [ ]:
R_LO, R_HI = 0.78, 0.88
ratio_A51 = 0.541223 * DWT**0.0390339          # Fig. A.51
delta = pd.Series({
    "Table 2.1: Delta band low [t]":   DWT / R_HI,
    "Table 2.1: Delta band high [t]":  DWT / R_LO,
    "Fig. A.51: DWT/Delta":            ratio_A51,
    "Fig. A.51: Delta [t]":            DWT / ratio_A51,
    "Fig. A.52: Delta [t]":            4773.95 + 1.1213 * DWT,
})
DELTA = delta["Fig. A.51: Delta [t]"]          # the value carried forward, as in class
print(f"Fig. A.52 against A.51: {100 * (delta['Fig. A.52: Delta [t]'] / DELTA - 1):+.1f} %")
delta.round(4)

## 2 · Tanker regressions with 95 % bands (Figs. A.53–A.58)
Formulas as printed on the figures (Kalokairinos et al. 2000–2005), all of the form a·DWT^b. The figures carry an **asterisk**: the outer purple curves are the 95 % prediction band. The band factors below were **read by eye** off the figures (about ±10–14 % on the dimensions, wider on the lightship); treat them as approximate.

In [ ]:
TANKER = {  # name: (a, b, figure, low factor, high factor, unit)
    "L_BP":  (6.86408,  0.307233, "A.53", 0.89, 1.11, "m"),
    "B":     (0.962394, 0.324898, "A.54", 0.88, 1.12, "m"),
    "D":     (0.45914,  0.330623, "A.55", 0.88, 1.14, "m"),
    "T":     (0.45011,  0.303134, "A.56", 0.87, 1.13, "m"),
    "LBD":   (3.05049,  0.962234, "A.57", 0.85, 1.17, "m3"),
    "LS":    (2.9186,   0.75548,  "A.58", 0.82, 1.20, "t"),
}
DATA_RANGE = (5_000, 550_000)   # deadweights covered by the tanker sample, read off the figures

def tanker_fits(dwt):
    rows = []
    for k, (a, b, fig, lo, hi, unit) in TANKER.items():
        v = a * dwt**b
        rows.append(dict(item=k, mean=v, band_low=v * lo, band_high=v * hi, unit=unit, figure=fig))
    return pd.DataFrame(rows).set_index("item")

est = tanker_fits(DWT)
if not (DATA_RANGE[0] <= DWT <= DATA_RANGE[1]):
    print(f"WARNING: DWT {DWT:,.0f} t is outside the fitted range: extrapolation.")
est.round(3)

## 3 · Block coefficient from the speed (Horn, Table 2.15)
The tanker fits give no C_B. The course takes it from Horn's formula for single-screw ships at service speed, C_B = 1.06 − 1.68·Fn, and no other formula.

In [ ]:
L, B, T, D = (est.loc[k, "mean"] for k in ["L_BP", "B", "T", "D"])
Fn = V_SERVICE * KN / math.sqrt(G * L)
CB = 1.06 - 1.68 * Fn
print(f"Fn = {Fn:.3f}   C_B (Horn) = {CB:.3f}")

## 4 · The checks of Lecture 1.4, and the owner's draught

In [ ]:
geo = RHO * CB * L * B * T
LS = est.loc["LS", "mean"]
checks = pd.Series({
    "1a  rho*CB*L*B*T [t]": geo,
    "1a  vs Delta [%]": 100 * (geo / DELTA - 1),
    "1b  Delta - DWT [t]": DELTA - DWT,
    "1b  LS fit vs Delta-DWT [%]": 100 * (LS / (DELTA - DWT) - 1),
    "1c  L*B*D vs fit A.57 [%]": 100 * (L * B * D / est.loc["LBD", "mean"] - 1),
    "2   L/B": L / B, "2   B/T": B / T, "2   B/D": B / D, "2   L/D": L / D, "2   Fn": Fn,
    "3   inside data range?": float(DATA_RANGE[0] <= DWT <= DATA_RANGE[1]),
    "4   T fit - limit [m]": T - T_MAX,
})
checks.round(3)

*Your answer:* does the set close? Which value would you change first, and why? Lecture 1.5, slide 4, closes the gap with the design equation; compare your answer with it.

## 5 · Length against deadweight

In [ ]:
d = np.linspace(5_000, 150_000, 200)
a, b, fig, lo, hi, _ = TANKER["L_BP"]
plt.figure(figsize=(8, 5))
plt.plot(d, a * d**b, "r-", label="Tankers, Fig. A.53 (LBP)")
plt.fill_between(d, lo * a * d**b, hi * a * d**b, color="purple", alpha=0.12, label="95 % prediction band (approx.)")
plt.axvline(DWT, color="k", ls=":", lw=1, label=f"{SHIP}: DWT {DWT:,.0f} t")
plt.xlabel("DWT [t]"); plt.ylabel("Length [m]"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Length against deadweight: Appendix A tankers"); plt.show()

Then take **Quiz 1**, the week's self-assessment (link from the lecturer).

---
## Optional · use this notebook for your project
The Ship Design Project is group work done outside class: your group applies these methods to its own Baseline ship for D2. This section is not part of the homework. It enters with TEU (a volume carrier, Table 1.9) at the brief's 10 t per TEU, uses the container fits of Appendix A (Figs. A.29–A.38, with approximate 95 % bands), the TEU-based Kristensen curves (Figs. A.47–A.50), the offshore supply family (Appendix A.14) for a PSV-type hull if your group takes it, and fits your own reference ships.

In [ ]:
# INPUT (project, optional): your group's ship
P_SHIP      = "My project ship"  # your group's Baseline ship
P_TEU       = None               # nominal capacity from your owner's brief, e.g. 500
P_T_PER_TEU = 10.0               # tonnes of payload per TEU: fixed by the brief (Fig. A.49 fleet average)
P_PAYLOAD_TO_DWT = 0.75          # Fig. A.49 (Kristensen): payload = 0.75 x DWT
P_V_SERVICE = None               # service speed from your owner's brief [kn]
P_HULL_FAMILY = "container"      # "container", or "psv" if your group takes the suggested PSV-type hull (Appendix A.14)

CONTAINER = {  # name: (formula f(DWT), figure, low factor, high factor, unit); bands read off the figures
    "Delta":  (lambda d: 1.77955 * d**0.975578,      "A.30", 0.88, 1.14, "t"),
    "L_BP":   (lambda d: 3.54132 * d**0.388442,      "A.31", 0.83, 1.19, "m"),
    "B":      (lambda d: 1.55219 * d**0.284381,      "A.32", 0.875, 1.13, "m"),
    "D":      (lambda d: 0.299394 * d**0.38902,      "A.33", 0.83, 1.22, "m"),
    "T":      (lambda d: -17.1581 + 2.72338 * math.log(d), "A.34", None, None, "m"),
    "CB":     (lambda d: 0.908683 * d**-0.0320423,   "A.38", 0.85, 1.17, "-"),
    "LS":     (lambda d: 0.973087 * d**0.91242,      "A.36", 0.65, 1.55, "t"),
    "DWT/Delta": (lambda d: 0.5608 * d**0.024,       "A.29", None, None, "-"),
}
C_T_BAND = (1.1, 1.3)            # m below / above the draught fit (Fig. A.34, logarithmic)
C_DATA_RANGE = (4000, 105000)

def container_fits(dwt):
    rows = []
    for k, (f, fig, lo, hi, unit) in CONTAINER.items():
        v = f(dwt)
        if k == "T":
            lo_v, hi_v = v - C_T_BAND[0], v + C_T_BAND[1]
        elif lo is None:
            lo_v = hi_v = float("nan")
        else:
            lo_v, hi_v = v * lo, v * hi
        rows.append(dict(item=k, mean=v, band_low=lo_v, band_high=hi_v, unit=unit, figure=fig))
    return pd.DataFrame(rows).set_index("item")

P_READY = P_TEU is not None and P_V_SERVICE is not None
if not P_READY:
    P_DWT = None
    print("Optional section: set P_TEU and P_V_SERVICE above to run it for your project ship.")
else:
    P_DWT = P_T_PER_TEU * P_TEU / P_PAYLOAD_TO_DWT
    print(f"{P_SHIP}: {P_TEU} TEU x {P_T_PER_TEU} t -> DWT = {P_DWT:,.0f} t")
    if not (C_DATA_RANGE[0] <= P_DWT <= C_DATA_RANGE[1]):
        print(f"WARNING: DWT {P_DWT:,.0f} t is outside the fitted range {C_DATA_RANGE[0]:,}–{C_DATA_RANGE[1]:,} t: extrapolation.")
    display(container_fits(P_DWT).round(3))

In [ ]:
# Fig. A.47: number of containers (TEU) against L_pp [m], digitized
A47_L   = np.array([60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300])
A47_TEU = np.array([100, 137, 210, 326, 420, 547, 662, 851, 1072, 1272, 1471, 1724, 1976, 2228, 2459, 2701, 2974, 3248, 3600, 4099, 4456, 4824, 5203, 5550, 5970])
# Fig. A.48: breadth and draught against L_pp, read by eye (about ±1 m)
A48_L = np.array([60, 85, 100, 135, 160, 185, 200, 235, 310])
A48_B = np.array([10.3, 13.0, 18.5, 23.3, 26.5, 30.0, 32.2, 32.2, 40.8])
A48_T = np.array([3.8, 5.8, 6.6, 8.6, 9.5, 10.2, 10.6, 11.3, 13.2])
# Fig. A.50: service speed [kn] against L_pp, read by eye (about ±0.5 kn)
A50_L = np.array([50, 90, 130, 170, 230, 290, 310])
A50_V = np.array([11.3, 14.4, 17.0, 19.7, 22.7, 25.3, 26.2])

if P_READY:
    L_k = float(np.interp(P_TEU, A47_TEU, A47_L))
    print(f"Fig. A.47: {P_TEU} TEU -> L_pp = {L_k:.0f} m")
    print(f"Fig. A.48: B = {np.interp(L_k, A48_L, A48_B):.1f} m, T = {np.interp(L_k, A48_L, A48_T):.1f} m")
    print(f"Fig. A.50: typical service speed at that length = {np.interp(L_k, A50_L, A50_V):.1f} kn (yours: {P_V_SERVICE} kn)")
else:
    print("Set P_TEU and P_V_SERVICE in the INPUT cell above to use the Kristensen curves.")

In [ ]:
def psv_family(dwt):
    Delta = 5.2301 * dwt**0.8752                         # Fig. A.136, R2 = 0.91
    nabla = Delta / RHO
    L = (nabla**(1/3) * 1.7841) ** (1 / (1 - 0.1884))    # Fig. A.134: L/nabla^(1/3) = 1.7841 L^0.1884
    LB = -0.0006*L**2 + 0.0851*L + 0.7841                # Fig. A.131
    BT = -0.0002*L**2 + 0.0195*L + 2.4877                # Fig. A.132
    CB = -1e-4*L**2 + 0.0116*L + 0.3878                  # Fig. A.135
    B = L / LB; T = B / BT
    return pd.Series({"Delta": Delta, "DWT/Delta": dwt / Delta, "L_BP": L, "B": B, "T": T, "CB": CB,
                      "rho*CB*L*B*T": RHO * CB * L * B * T,
                      "Fn": P_V_SERVICE * KN / math.sqrt(G * L)})

psv_family(P_DWT).round(3) if (P_READY and P_HULL_FAMILY == "psv") else print("Set P_TEU, P_V_SERVICE and P_HULL_FAMILY = 'psv' to use this family.")

**Your reference ship database (D2).** Create `reference_ships.csv` next to this notebook with one row per similar ship and the columns `name, year, source, TEU, DWT, LOA, LBP, B, D, T, V, P_kW`. Rules for D2 are in 0.2 Ship Design Project. The cell below fits your own regression L = a·DWT^b in log–log form, with its 95 % prediction band.

In [ ]:
import os
from math import sqrt

def fit_power(x, y, x_new):
    """Fit y = a*x^b by least squares on logs; return a, b, R2 and the 95 % prediction band at x_new."""
    X, Y = np.log(np.asarray(x, float)), np.log(np.asarray(y, float))
    n = len(X)
    b, lna = np.polyfit(X, Y, 1)
    resid = Y - (lna + b * X)
    s = sqrt((resid @ resid) / (n - 2))
    R2 = 1 - (resid @ resid) / ((Y - Y.mean()) @ (Y - Y.mean()))
    from scipy import stats
    t = stats.t.ppf(0.975, n - 2)
    x0 = np.log(np.asarray(x_new, float))
    se = s * np.sqrt(1 + 1/n + (x0 - X.mean())**2 / ((X - X.mean()) @ (X - X.mean())))
    mean = lna + b * x0
    return math.exp(lna), b, R2, np.exp(mean), np.exp(mean - t*se), np.exp(mean + t*se)

if os.path.exists("reference_ships.csv"):
    db = pd.read_csv("reference_ships.csv")
    db = db.dropna(subset=["DWT", "LBP"])
    print(f"{len(db)} ships with DWT and LBP")
    if len(db) >= 4 and P_READY:
        a, b, R2, m, lo, hi = fit_power(db.DWT, db.LBP, [P_DWT])
        print(f"Your fit: L_BP = {a:.3f} * DWT^{b:.4f}   R2 = {R2:.3f}")
        print(f"At DWT {P_DWT:,.0f} t: L_BP = {m[0]:.1f} m, 95 % band {lo[0]:.1f}–{hi[0]:.1f} m")
        print(f"Appendix A (Fig. A.31): {container_fits(P_DWT).loc['L_BP','mean']:.1f} m")
    else:
        print("Add at least 4 ships, and set P_TEU and P_V_SERVICE, to fit a regression.")
else:
    db = None
    print("No reference_ships.csv yet.")